# GameTheory 24b : Le temoin d'impossibilite

> **Grain1 d'ai-01 sur #12205 §6** : Robinson-Goforth (#12364, GT-24 chemin minimal) porte les points 1-3 du critere d'acceptance mais pas le **point 4** (temoin d'impossibilite). Ce notebook ferme le point 4 en s'appuyant sur la **structure produit** des chambres : `d_chambre(G,H) = d_perm(row_G,row_H) + d_perm(col_G,col_H)`.

**Conventions reprises de GT-24** : `canonique`, `swap_valeurs_adjacentes`, `swap_jeu(cote, k)`, 24 ordres stricts sur 4 cases, 576 chambres, 6 voisins par jeu (3 swaps x {ligne, colonne}).

## 0. Configuration

In [1]:
# === Configuration : GameTheory 24b ===
from itertools import permutations
from collections import deque, Counter

print("GameTheory 24b : temoin d'impossibilite certifie sur R-G (point 4)")
print()

def canonique(t):
    vals = sorted(set(t))
    return tuple(vals.index(v) + 1 for v in t)

def swap_valeurs_adjacentes(t, k):
    pk, pk1 = t.index(k), t.index(k + 1)
    l = list(t)
    l[pk], l[pk1] = l[pk1], l[pk]
    return tuple(l)

def swap_jeu(jeu, cote, k):
    row, col = jeu
    if cote == "ligne":
        return (swap_valeurs_adjacentes(row, k), col)
    return (row, swap_valeurs_adjacentes(col, k))

def adj_chambre(g):
    return [swap_jeu(g, cote, k) for cote in ("ligne", "colonne") for k in (1, 2, 3)]

def adj_perm(t):
    return {swap_valeurs_adjacentes(t, k) for k in (1, 2, 3)}

def bfs(depart, voisins_fn):
    dist = {depart: 0}
    q = deque([depart])
    while q:
        u = q.popleft()
        for v in voisins_fn(u):
            if v not in dist:
                dist[v] = dist[u] + 1
                q.append(v)
    return dist

stricts = sorted({canonique(t) for t in permutations(range(1, 5))})
chambres = [(r, c) for r in stricts for c in stricts]
print(f"Permutaedre : {len(stricts)} ordres stricts, {len(adj_perm(stricts[0]))} voisins / ordre")
print(f"Chambres : {len(chambres)} sommets, {len(adj_chambre(chambres[0]))} voisins / chambre")


GameTheory 24b : temoin d'impossibilite certifie sur R-G (point 4)

Permutaedre : 24 ordres stricts, 3 voisins / ordre
Chambres : 576 sommets, 6 voisins / chambre


## 1. La structure produit

Le graphe des chambres est un **produit cartesien** de deux copies du permutaedre `S_4`. Les generateurs ne touchent qu'un seul cote a la fois, donc la distance dans le produit est exactement la somme des distances dans chaque facteur.

In [2]:
# === Section 1.1 : verification exhaustive ===
IDENTITE = ((1, 2, 3, 4), (1, 2, 3, 4))
dist_perm = {o: bfs(o, adj_perm) for o in stricts}
dist_chambre = bfs(IDENTITE, lambda g: adj_chambre(g))

ecarts = []
for (r, c) in chambres:
    d_ch = dist_chambre[(r, c)]
    d_calc = dist_perm[IDENTITE[0]][r] + dist_perm[IDENTITE[1]][c]
    if d_ch != d_calc:
        ecarts.append(((r, c), d_ch, d_calc))

print(f"Verification exhaustive sur 576 chambres : {len(ecarts)} ecarts")
print(f"Structure produit : {'EXACTE' if not ecarts else 'NON'}")
print(f"Diametre permutaedre : {max(max(d.values()) for d in dist_perm.values())}")
print(f"Diametre chambres : {max(dist_chambre.values())}")
assert not ecarts
print("ASSERTION PASS : structure produit verifiee sur 576/576")


Verification exhaustive sur 576 chambres : 0 ecarts
Structure produit : EXACTE
Diametre permutaedre : 6
Diametre chambres : 12
ASSERTION PASS : structure produit verifiee sur 576/576


## 2. Le certificat analytique

Pour `(G, H, k_max)` :

- **`IMPOSSIBLE`** : `d_row + d_col > k_max`
- **`POSSIBLE`** : `d_row + d_col <= k_max`

Verification de la table 24x24 (symetrique, triangulaire) :

In [3]:
# === Section 2.1 : precalcul 24x24 ===
D = {(o1, o2): dist_perm[o1][o2] for o1 in stricts for o2 in stricts}
non_sym = [(o1, o2) for o1 in stricts for o2 in stricts if D[(o1,o2)] != D[(o2,o1)]]
print(f"Symetrie 24x24 : {len(non_sym)} asymetries")

import itertools
violations = sum(1 for o1, o2, o3 in itertools.product(stricts, repeat=3)
                 if D[(o1, o3)] > D[(o1, o2)] + D[(o2, o3)])
print(f"Inegalite triangulaire : {violations} violations sur {24**3} triplets")
assert violations == 0
print("ASSERTION PASS : table 24x24 symetrique et triangulaire")


Symetrie 24x24 : 0 asymetries
Inegalite triangulaire : 0 violations sur 13824 triplets
ASSERTION PASS : table 24x24 symetrique et triangulaire


## 3. Le temoin en action

- **Cas A** : antipode `IDENTITE -> RENVERS`, k_max=5 -> IMPOSSIBLE
- **Cas B** : row identique, col antipode, k_max=3 -> IMPOSSIBLE
- **Cas C** : frontiere POSSIBLE, k_max=1 -> POSSIBLE

In [4]:
# === Section 3.1 : certifier_impossibilite ===
def certifier_impossibilite(G, H, k_max, table=D):
    d_row = table[(G[0], H[0])]
    d_col = table[(G[1], H[1])]
    d_min = d_row + d_col
    if d_min > k_max:
        return {'verdict': 'IMPOSSIBLE', 'd_row': d_row, 'd_col': d_col,
                'd_min': d_min, 'k_max': k_max,
                'preuve': f'd_perm(row) + d_perm(col) = {d_row} + {d_col} = {d_min} > {k_max}'}
    else:
        return {'verdict': 'POSSIBLE', 'd_row': d_row, 'd_col': d_col,
                'd_min': d_min, 'k_max': k_max,
                'preuve': f'd_perm(row) + d_perm(col) = {d_row} + {d_col} = {d_min} <= {k_max}'}

RENVERS = ((4, 3, 2, 1), (4, 3, 2, 1))
cas_A = certifier_impossibilite(IDENTITE, RENVERS, 5)
print("CAS A (antipode, k_max=5) :", cas_A['verdict'], "-", cas_A['preuve'])

H_B = ((1, 2, 3, 4), (3, 4, 1, 2))
cas_B = certifier_impossibilite(IDENTITE, H_B, 3)
print("CAS B (row identique, k_max=3) :", cas_B['verdict'], "-", cas_B['preuve'])

H_C = ((1, 2, 4, 3), (1, 2, 3, 4))
cas_C = certifier_impossibilite(IDENTITE, H_C, 1)
print("CAS C (frontiere, k_max=1) :", cas_C['verdict'], "-", cas_C['preuve'])


CAS A (antipode, k_max=5) : IMPOSSIBLE - d_perm(row) + d_perm(col) = 6 + 6 = 12 > 5
CAS B (row identique, k_max=3) : IMPOSSIBLE - d_perm(row) + d_perm(col) = 0 + 4 = 4 > 3
CAS C (frontiere, k_max=1) : POSSIBLE - d_perm(row) + d_perm(col) = 1 + 0 = 1 <= 1


## 4. Verification BFS tronque

Pour chaque cas IMPOSSIBLE, on verifie par BFS tronque que `H` n'est **jamais** dans les sommets a distance `<= k_max` de `G`.

In [5]:
# === Section 4.1 : BFS tronque ===
def bfs_tronque(depart, voisins_fn, profondeur_max):
    atteints = {depart}
    frontiere = {depart}
    for d in range(profondeur_max):
        nouvelle_frontiere = set()
        for u in frontiere:
            for v in voisins_fn(u):
                if v not in atteints:
                    atteints.add(v)
                    nouvelle_frontiere.add(v)
        frontiere = nouvelle_frontiere
        if not frontiere:
            break
    return atteints

atte_A = bfs_tronque(IDENTITE, lambda g: adj_chambre(g), 5)
verif_A = RENVERS not in atte_A
print(f"CAS A : sommets a <= 5 pas = {len(atte_A)}, RENVERS present ? {RENVERS in atte_A}")
print(f"  Temoin IMPOSSIBLE verifie : {verif_A}")

atte_B = bfs_tronque(IDENTITE, lambda g: adj_chambre(g), 3)
verif_B = H_B not in atte_B
print(f"CAS B : sommets a <= 3 pas = {len(atte_B)}, H_B present ? {H_B in atte_B}")
print(f"  Temoin IMPOSSIBLE verifie : {verif_B}")

atte_C = bfs_tronque(IDENTITE, lambda g: adj_chambre(g), 1)
verif_C = H_C in atte_C
print(f"CAS C : sommets a <= 1 pas = {len(atte_C)}, H_C present ? {H_C in atte_C}")
print(f"  Temoin POSSIBLE verifie : {verif_C}")

assert verif_A and verif_B and verif_C
print("ASSERTION PASS : 3 cas (2 IMPOSSIBLE, 1 POSSIBLE) verifies")


CAS A : sommets a <= 5 pas = 235, RENVERS present ? False
  Temoin IMPOSSIBLE verifie : True
CAS B : sommets a <= 3 pas = 68, H_B present ? False
  Temoin IMPOSSIBLE verifie : True
CAS C : sommets a <= 1 pas = 7, H_C present ? True
  Temoin POSSIBLE verifie : True
ASSERTION PASS : 3 cas (2 IMPOSSIBLE, 1 POSSIBLE) verifies


## 5. Inventaire -- combien de triplets sont IMPOSSIBLE ?

Echantillon de 100 chambres representatives, balayage des `k_max ∈ {0..11}`.

In [6]:
# === Section 5.1 : inventaire ===
import random
random.seed(42)
echantillon = random.sample(chambres, 100)

resultats_par_k = {}
for k_max in range(0, 12):
    compteur_imp = 0
    compteur_tot = 0
    for G in echantillon:
        for H in chambres:
            if G == H:
                continue
            d_min = D[(G[0], H[0])] + D[(G[1], H[1])]
            compteur_tot += 1
            if d_min > k_max:
                compteur_imp += 1
    resultats_par_k[k_max] = (compteur_imp, compteur_tot)

print("k_max | IMPOSSIBLE / Total | %")
print("-" * 50)
for k_max in (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11):
    imp, tot = resultats_par_k[k_max]
    pct = 100.0 * imp / tot
    print(f"  {k_max:>2}  | {imp:>6} / {tot:>6} | {pct:>5.1f}%")

imp5, tot5 = resultats_par_k[5]
print()
print(f"A k_max=5 : {100*imp5/tot5:.1f}% des paires sont IMPOSSIBLE -- le temoin tranche souvent")


k_max | IMPOSSIBLE / Total | %
--------------------------------------------------
   0  |  57500 /  57500 | 100.0%
   1  |  56900 /  57500 |  99.0%
   2  |  55000 /  57500 |  95.7%
   3  |  50800 /  57500 |  88.3%
   4  |  43700 /  57500 |  76.0%
   5  |  34100 /  57500 |  59.3%
   6  |  23500 /  57500 |  40.9%
   7  |  13900 /  57500 |  24.2%
   8  |   6800 /  57500 |  11.8%
   9  |   2600 /  57500 |   4.5%
  10  |    700 /  57500 |   1.2%
  11  |    100 /  57500 |   0.2%

A k_max=5 : 59.3% des paires sont IMPOSSIBLE -- le temoin tranche souvent


## 6. Frontiere et bornes

**Certifie** : pour `k_max < 12`, IMPOSSIBLE est certain.

**Non certifie** : un cas POSSIBLE garantit l'existence, pas un chemin explicite (le constructeur de GT-24 produit le chemin). Au-dela de `k_max = 12`, tout est POSSIBLE.

In [7]:
# === Section 6.1 : resume final ===
print("RESUME -- Le point 4 du critere #12205 est TENU sur Robinson-Goforth")
print()
print("Verdict par cas :")
print(f"  CAS A : {cas_A['verdict']}")
print(f"  CAS B : {cas_B['verdict']}")
print(f"  CAS C : {cas_C['verdict']}")
print()
print("Verification par BFS tronque :")
print(f"  CAS A IMPOSSIBLE verifie : {verif_A}")
print(f"  CAS B IMPOSSIBLE verifie : {verif_B}")
print(f"  CAS C POSSIBLE verifie   : {verif_C}")
print()
print(f"Structure produit : 576/576 chambres, 0 ecart")
print(f"Table 24x24 : symetrique et triangulaire")
print(f"A k_max=5 : {100*imp5/tot5:.1f}% des paires sont IMPOSSIBLE")


RESUME -- Le point 4 du critere #12205 est TENU sur Robinson-Goforth

Verdict par cas :
  CAS A : IMPOSSIBLE
  CAS B : IMPOSSIBLE
  CAS C : POSSIBLE

Verification par BFS tronque :
  CAS A IMPOSSIBLE verifie : True
  CAS B IMPOSSIBLE verifie : True
  CAS C POSSIBLE verifie   : True

Structure produit : 576/576 chambres, 0 ecart
Table 24x24 : symetrique et triangulaire
A k_max=5 : 59.3% des paires sont IMPOSSIBLE


## Exercices

### Exercice 1 -- Un autre cas IMPOSSIBLE

Choisir un autre couple `(G, H)` et un `k_max` tels que le temoin certifie IMPOSSIBLE. Verifier par BFS tronque.

In [8]:
# Exercice 1 : choisir un cas IMPOSSIBLE et verifier
G_exo1 = ((1, 2, 3, 4), (1, 2, 3, 4))  # Identite
H_exo1 = ((4, 3, 2, 1), (1, 2, 3, 4))  # row antipode, col identique -> d_min = 6
k_max_exo1 = 3                           # IMPOSSIBLE : 6 > 3
cert = certifier_impossibilite(G_exo1, H_exo1, k_max_exo1)
print("Certificat Exercice 1 :", cert)
atteints = bfs_tronque(G_exo1, lambda g: adj_chambre(g), k_max_exo1)
verif = H_exo1 not in atteints
print(f"BFS tronque : H dans sommets a <= {k_max_exo1} pas ? {H_exo1 in atteints}")
print(f"Verifie : {verif}")
# resultat = None  # pour exercice etudiant


Certificat Exercice 1 : {'verdict': 'IMPOSSIBLE', 'd_row': 6, 'd_col': 0, 'd_min': 6, 'k_max': 3, 'preuve': 'd_perm(row) + d_perm(col) = 6 + 0 = 6 > 3'}
BFS tronque : H dans sommets a <= 3 pas ? False
Verifie : True


### Exercice 2 -- Trouver le plus petit k_max POSSIBLE pour un antipode

Pour `(IDENTITE, RENVERSEMENT)`, trouver le plus petit `k_max` tel que `certifier_impossibilite` rend POSSIBLE.

In [9]:
# Exercice 2 : trouver la distance de l'antipode
G = IDENTITE
H = RENVERS
resultat = None  # TODO etudiant
for k in range(0, 13):
    c = certifier_impossibilite(G, H, k)
    if c['verdict'] == 'POSSIBLE':
        resultat = k
        print(f"Plus petit k_max rendant POSSIBLE : {k} (distance = {c['d_min']})")
        break


Plus petit k_max rendant POSSIBLE : 12 (distance = 12)


### Exercice 3 -- Trois paires d_row=0 et d_col=6

Trouver 3 paires distinctes avec `d_row=0` (row identique) et `d_col=6` (antipode colonne).

In [10]:
# Exercice 3 : trois paires d_row=0 et d_col=6
candidates = []
for G in chambres:
    for H in chambres:
        if G == H:
            continue
        if D[(G[0], H[0])] == 0 and D[(G[1], H[1])] == 6:
            candidates.append((G, H))
            if len(candidates) >= 3:
                break
    if len(candidates) >= 3:
        break

resultat = None  # pour exercice etudiant
for i, (G, H) in enumerate(candidates, 1):
    c5 = certifier_impossibilite(G, H, 5)
    c6 = certifier_impossibilite(G, H, 6)
    print(f"Paire {i} : k_max=5 -> {c5['verdict']}, k_max=6 -> {c6['verdict']}")


Paire 1 : k_max=5 -> IMPOSSIBLE, k_max=6 -> POSSIBLE
Paire 2 : k_max=5 -> IMPOSSIBLE, k_max=6 -> POSSIBLE
Paire 3 : k_max=5 -> IMPOSSIBLE, k_max=6 -> POSSIBLE


## Conclusion -- le point 4 sur Robinson-Goforth est tenu

**Acceptance #12205 §5** :

1. Substrat dont le verificateur existe sur main : GT-24 porte les points 1-3. **OUI**.
2. Generateur != verificateur : `certifier_impossibilite` (O(1)) != BFS exhaustif (O(576^k_max)). **OUI**.
3. Temoin certifie : verdict IMPOSSIBLE porte certificat analytique (d_row, d_col, k_max). **OUI**.
4. Cas sans solution = temoin d'impossibilite, pas silence : section 4 verifie par BFS tronque. Section 5 : plus de la moitie des paires a k_max=5 sont IMPOSSIBLE. **OUI**.

**Substrats independants portant les 4 points** :
- Life/Conway (PR #14205, #12395)
- AMD (`impossibility_strict_payments`)
- Robinson-Goforth (ce notebook)
- Tweety-5d (`afB_no_stable`, PR #13628)

Reference croisee : #12205 - #12364 - #13628.